# NIDS v1 Baseline — Intentionally Vulnerable

> **Read this first.** This notebook is kept in the portfolio as a deliberate baseline, not as a finished result.
>
> It was the first pass at an NSL-KDD intrusion-detection pipeline. While building it I shipped a handful of mistakes that are extremely common in ML notebooks but that no SAST tool catches:
>
> - **Data leakage** — `difficulty_level` is kept as a feature even though it correlates with the label, and SMOTE is applied before the train/val split
> - **Insecure deserialization** — `joblib.load(...)` of artifacts with no integrity check
> - **Supply-chain rot** — `!pip install ... -q` with no pinned versions, `!wget` from raw GitHub with no checksum
> - **Model evadability** — the trained LSTM flips on ε ≤ 0.05 FGSM perturbation for the majority of attack samples
>
> Those mistakes were the motivation for the sibling project [`mlsecops-agent/`](mlsecops-agent/) — a Claude-driven audit agent that surfaces exactly this class of issue. The fixed pipeline lives in [`nids_pipeline_v2.ipynb`](nids_pipeline_v2.ipynb).
>
> **Outputs have been stripped** to keep the repo small. The notebook is reproducible against the NSL-KDD dataset (see v2 for the cleaned setup).


In [ ]:
!wget -q https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt
!wget -q https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest+.txt
!pip install imbalanced-learn -q
print("✅ Dataset téléchargé !")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')
from collections import Counter
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from imblearn.over_sampling import SMOTE

print("✅ Imports OK !")

In [ ]:
columns = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
    'wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty_level'
]

train_df = pd.read_csv('KDDTrain+.txt', header=None, names=columns)
test_df  = pd.read_csv('KDDTest+.txt',  header=None, names=columns)

train_df.drop('difficulty_level', axis=1, inplace=True)
test_df.drop('difficulty_level',  axis=1, inplace=True)

print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)
train_df.head()

In [ ]:
attack_map = {
    'normal':'Normal',
    # DoS
    'back':'DoS','land':'DoS','neptune':'DoS','pod':'DoS','smurf':'DoS',
    'teardrop':'DoS','apache2':'DoS','udpstorm':'DoS','processtable':'DoS','worm':'DoS',
    # Probe
    'satan':'Probe','ipsweep':'Probe','nmap':'Probe','portsweep':'Probe',
    'mscan':'Probe','saint':'Probe',
    # R2L
    'guess_passwd':'R2L','ftp_write':'R2L','imap':'R2L','phf':'R2L','multihop':'R2L',
    'warezmaster':'R2L','warezclient':'R2L','spy':'R2L','xlock':'R2L','xsnoop':'R2L',
    'snmpguess':'R2L','snmpgetattack':'R2L','httptunnel':'R2L','sendmail':'R2L','named':'R2L',
    # U2R
    'buffer_overflow':'U2R','loadmodule':'U2R','perl':'U2R','rootkit':'U2R',
    'mailbomb':'U2R','ps':'U2R','sqlattack':'U2R','xterm':'U2R'
}

train_df['category'] = train_df['label'].map(attack_map).fillna('Unknown')
test_df['category']  = test_df['label'].map(attack_map).fillna('Unknown')

train_df = train_df[train_df['category'] != 'Unknown'].reset_index(drop=True)
test_df  = test_df[test_df['category']   != 'Unknown'].reset_index(drop=True)

print("Distribution des catégories :")
print(train_df['category'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

cat_counts = train_df['category'].value_counts()
colors = ['#2ecc71','#e74c3c','#3498db','#f39c12','#9b59b6']

axes[0].bar(cat_counts.index, cat_counts.values,
            color=colors[:len(cat_counts)], edgecolor='black')
axes[0].set_title('Distribution des 5 catégories', fontsize=13)
axes[0].set_xlabel('Catégorie')
axes[0].set_ylabel("Nombre d'entrées")
for i, v in enumerate(cat_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontsize=9)

axes[1].pie(cat_counts.values,
            labels=cat_counts.index,
            autopct='%1.1f%%',
            colors=colors[:len(cat_counts)],
            startangle=90)
axes[1].set_title('Répartition par catégorie (%)', fontsize=13)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

In [ ]:
le_temp = LabelEncoder()
train_df['label_encoded'] = le_temp.fit_transform(train_df['category'])

numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'label_encoded']

corr_with_target = train_df[numeric_cols + ['label_encoded']].corr()['label_encoded'].drop('label_encoded')
top_features = corr_with_target.abs().sort_values(ascending=False).head(15).index.tolist()

plt.figure(figsize=(13, 10))
corr_matrix = train_df[top_features + ['label_encoded']].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, annot_kws={"size": 7})
plt.title('Heatmap de corrélation — Top 15 features', fontsize=13)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

train_df.drop('label_encoded', axis=1, inplace=True)

In [ ]:
selector = VarianceThreshold(threshold=0.01)
selector.fit(train_df[numeric_cols])

low_var_features = [numeric_cols[i] for i, s in enumerate(selector.get_support()) if not s]
print(f"Features à variance quasi-nulle ({len(low_var_features)}) :")
print(low_var_features)

variances = train_df[numeric_cols].var().sort_values()
plt.figure(figsize=(10, 6))
variances.head(20).plot(kind='barh', color='#3498db')
plt.title('20 features avec la plus faible variance')
plt.xlabel('Variance')
plt.tight_layout()
plt.savefig('low_variance_features.png', dpi=150)
plt.show()

In [ ]:
# Protocoles par catégorie
proto_analysis = train_df.groupby(['protocol_type','category']).size().unstack(fill_value=0)
print(proto_analysis)

proto_analysis.plot(kind='bar', figsize=(10, 6), colormap='tab10', edgecolor='black')
plt.title('Protocoles par catégorie de trafic')
plt.xlabel('Protocol Type')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Catégorie', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('protocol_distribution.png', dpi=150)
plt.show()

# Top 5 services par catégorie d'attaque
for cat in ['DoS', 'Probe', 'R2L', 'U2R']:
    top_s = train_df[train_df['category'] == cat]['service'].value_counts().head(5)
    print(f"\nTop 5 services — {cat}:")
    print(top_s)

In [ ]:
df_train = train_df.copy()
df_test  = test_df.copy()

cat_features = ['protocol_type', 'service', 'flag']
df_train = pd.get_dummies(df_train, columns=cat_features)
df_test  = pd.get_dummies(df_test,  columns=cat_features)

df_train, df_test = df_train.align(df_test, join='left', axis=1, fill_value=0)

print(f"✅ Shape après One-Hot — Train: {df_train.shape} | Test: {df_test.shape}")

In [ ]:
drop_cols    = ['label', 'category']
feature_cols = [c for c in df_train.columns if c not in drop_cols]

X_train_raw = df_train[feature_cols].astype(float).values
X_test_raw  = df_test[feature_cols].astype(float).values

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

print("X_train shape:", X_train_scaled.shape)
print("X_test  shape:", X_test_scaled.shape)

In [ ]:
from collections import Counter
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder

# ── Label Encoding 5 classes ───────────────────────────────────
le = LabelEncoder()
le.fit(['DoS', 'Normal', 'Probe', 'R2L', 'U2R'])
y_train_cat = le.transform(df_train['category'])
y_test_cat  = le.transform(df_test['category'].map(
    lambda x: x if x in le.classes_ else 'Normal'))

print("Classes :", le.classes_)

# ── SMOTE ──────────────────────────────────────────────────────
before         = Counter(y_train_cat)
majority_count = max(before.values())
print(f"\nNombre cible par classe : {majority_count}")
print("\nDistribution AVANT SMOTE :")
for cls, count in sorted(before.items()):
    print(f"  {le.classes_[cls]:10s} : {count}")

sampling_strategy = {
    cls: majority_count
    for cls, count in before.items()
    if count < majority_count
}

smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42, k_neighbors=3)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train_cat)

after = Counter(y_train_smote)
print("\nDistribution APRÈS SMOTE :")
for cls, count in sorted(after.items()):
    print(f"  {le.classes_[cls]:10s} : {count}")

# ── Visualisation ──────────────────────────────────────────────
fig, axes   = plt.subplots(1, 2, figsize=(14, 5))
colors      = ['#e74c3c','#2ecc71','#3498db','#f39c12','#9b59b6']
class_names = le.classes_

before_vals = [before[i] for i in range(len(class_names))]
after_vals  = [after[i]  for i in range(len(class_names))]

axes[0].bar(class_names, before_vals, color=colors, edgecolor='black')
axes[0].set_title('Avant SMOTE')
axes[0].set_ylabel('Count')
for i, v in enumerate(before_vals):
    axes[0].text(i, v + 50, str(v), ha='center', fontsize=8)

axes[1].bar(class_names, after_vals, color=colors, edgecolor='black')
axes[1].set_title('Après SMOTE')
axes[1].set_ylabel('Count')
for i, v in enumerate(after_vals):
    axes[1].text(i, v + 50, str(v), ha='center', fontsize=8)

plt.suptitle('SMOTE — Toutes les classes égales', fontsize=13)
plt.tight_layout()
plt.savefig('smote_balance.png', dpi=150)
plt.show()

In [ ]:
numeric_only = df_train[feature_cols].select_dtypes(include=[np.number]).copy()
numeric_only['label_enc'] = y_train_cat

correlations = numeric_only.corr()['label_enc'].drop('label_enc').abs()
top3 = correlations.sort_values(ascending=False).head(3)

print("=== TOP 3 FEATURES CRITIQUES ===")
print(top3)

plt.figure(figsize=(7, 4))
top3.plot(kind='bar', color=['#e74c3c','#f39c12','#3498db'], edgecolor='black')
plt.title('Top 3 Features les plus corrélées avec les attaques')
plt.ylabel('Corrélation absolue')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('top3_features.png', dpi=150)
plt.show()

print("\n✅ Task 1 complète !")
print("Pipeline : One-Hot → StandardScaler → SMOTE (5 classes égales)")

In [ ]:
# ── Installations ──────────────────────────────────────────────
!wget -q https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt
!wget -q https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest+.txt
!pip install imbalanced-learn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')
from collections import Counter
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE

# Colonnes
columns = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
    'wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty_level'
]

# Chargement
train_df = pd.read_csv('KDDTrain+.txt', header=None, names=columns)
test_df  = pd.read_csv('KDDTest+.txt',  header=None, names=columns)
train_df.drop('difficulty_level', axis=1, inplace=True)
test_df.drop('difficulty_level',  axis=1, inplace=True)

# Mapping
attack_map = {
    'normal':'Normal',
    'back':'DoS','land':'DoS','neptune':'DoS','pod':'DoS','smurf':'DoS',
    'teardrop':'DoS','apache2':'DoS','udpstorm':'DoS','processtable':'DoS','worm':'DoS',
    'satan':'Probe','ipsweep':'Probe','nmap':'Probe','portsweep':'Probe',
    'mscan':'Probe','saint':'Probe',
    'guess_passwd':'R2L','ftp_write':'R2L','imap':'R2L','phf':'R2L','multihop':'R2L',
    'warezmaster':'R2L','warezclient':'R2L','spy':'R2L','xlock':'R2L','xsnoop':'R2L',
    'snmpguess':'R2L','snmpgetattack':'R2L','httptunnel':'R2L','sendmail':'R2L','named':'R2L',
    'buffer_overflow':'U2R','loadmodule':'U2R','perl':'U2R','rootkit':'U2R',
    'mailbomb':'U2R','ps':'U2R','sqlattack':'U2R','xterm':'U2R'
}

train_df['category'] = train_df['label'].map(attack_map).fillna('Unknown')
test_df['category']  = test_df['label'].map(attack_map).fillna('Unknown')
train_df = train_df[train_df['category'] != 'Unknown'].reset_index(drop=True)
test_df  = test_df[test_df['category']   != 'Unknown'].reset_index(drop=True)

# One-Hot Encoding
cat_features = ['protocol_type', 'service', 'flag']
df_train = pd.get_dummies(train_df, columns=cat_features)
df_test  = pd.get_dummies(test_df,  columns=cat_features)
df_train, df_test = df_train.align(df_test, join='left', axis=1, fill_value=0)

# Features
drop_cols    = ['label', 'category']
feature_cols = [c for c in df_train.columns if c not in drop_cols]
X_train_raw  = df_train[feature_cols].astype(float).values
X_test_raw   = df_test[feature_cols].astype(float).values

# StandardScaler
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

# Label Encoding
le = LabelEncoder()
le.fit(['DoS', 'Normal', 'Probe', 'R2L', 'U2R'])
y_train_cat = le.transform(df_train['category'])
y_test_cat  = le.transform(df_test['category'].map(
    lambda x: x if x in le.classes_ else 'Normal'))

# SMOTE toutes classes égales
before         = Counter(y_train_cat)
majority_count = max(before.values())
sampling_strategy = {
    cls: majority_count
    for cls, count in before.items()
    if count < majority_count
}
smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42, k_neighbors=3)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train_cat)

print("Distribution APRÈS SMOTE :")
after = Counter(y_train_smote)
for cls, count in sorted(after.items()):
    print(f"  {le.classes_[cls]:10s} : {count}")

# Labels binaires
y_train = np.array([0 if le.classes_[c] == 'Normal' else 1 for c in y_train_smote])
y_test  = np.array([0 if le.classes_[c] == 'Normal' else 1 for c in y_test_cat])

# Sauvegarde
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(le,     'label_encoder.pkl')
np.save('X_train_final.npy',  X_train_smote)
np.save('X_test_final.npy',   X_test_scaled)
np.save('y_train_final.npy',  y_train)
np.save('y_test_final.npy',   y_test)
np.save('y_train_cat.npy',    y_train_smote)
np.save('y_test_cat.npy',     y_test_cat)
np.save('feature_cols.npy',   np.array(feature_cols))

print("\n✅ Preprocessing terminé !")
print(f"   X_train : {X_train_smote.shape}")
print(f"   X_test  : {X_test_scaled.shape}")

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, Flatten, Dense,
                                      Dropout, LSTM, BatchNormalization)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import (confusion_matrix, classification_report,
                              accuracy_score, precision_score,
                              recall_score, f1_score)

print("✅ TensorFlow:", tf.__version__)

In [ ]:
X_train_3d = X_train_smote.reshape(X_train_smote.shape[0], X_train_smote.shape[1], 1)
X_test_3d  = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

print("X_train 3D:", X_train_3d.shape)
print("X_test  3D:", X_test_3d.shape)

In [ ]:
def build_cnn(input_shape):
    model = Sequential([
        Conv1D(filters=64, kernel_size=3, activation='relu',
               padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        Dropout(0.3),

        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.4),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

cnn_model = build_cnn((X_train_3d.shape[1], 1))
cnn_model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

cnn_history = cnn_model.fit(
    X_train_3d, y_train,
    epochs=30,
    batch_size=256,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)
print("✅ CNN entraîné !")

In [ ]:
def plot_history(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history['loss'],
                 label='Train Loss', color='#3498db', linewidth=2)
    axes[0].plot(history.history['val_loss'],
                 label='Val Loss', color='#e74c3c', linewidth=2)
    axes[0].set_title(f'{model_name} — Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history['accuracy'],
                 label='Train Accuracy', color='#2ecc71', linewidth=2)
    axes[1].plot(history.history['val_accuracy'],
                 label='Val Accuracy', color='#f39c12', linewidth=2)
    axes[1].set_title(f'{model_name} — Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle(f'Training History — {model_name}', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{model_name}_history.png', dpi=150)
    plt.show()

plot_history(cnn_history, 'CNN')

In [ ]:
def build_lstm(input_shape):
    model = Sequential([
        LSTM(128, return_sequences=True, input_shape=input_shape),
        BatchNormalization(),
        Dropout(0.3),

        LSTM(64, return_sequences=False),
        BatchNormalization(),
        Dropout(0.3),

        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

lstm_model = build_lstm((X_train_3d.shape[1], 1))
lstm_model.summary()

In [ ]:
lstm_history = lstm_model.fit(
    X_train_3d, y_train,
    epochs=30,
    batch_size=256,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)
print("✅ LSTM entraîné !")

In [ ]:
plot_history(lstm_history, 'LSTM')

In [ ]:
y_pred_cnn_prob  = cnn_model.predict(X_test_3d, verbose=0)
y_pred_cnn       = (y_pred_cnn_prob > 0.5).astype(int).flatten()

y_pred_lstm_prob = lstm_model.predict(X_test_3d, verbose=0)
y_pred_lstm      = (y_pred_lstm_prob > 0.5).astype(int).flatten()

print(f"CNN  — Attaques détectées : {y_pred_cnn.sum()}  / {len(y_pred_cnn)}")
print(f"LSTM — Attaques détectées : {y_pred_lstm.sum()} / {len(y_pred_lstm)}")

In [ ]:
def plot_confusion_matrix(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Attack'],
                yticklabels=['Normal', 'Attack'])
    plt.title(f'Confusion Matrix — {model_name}', fontsize=13)
    plt.ylabel('Réel')
    plt.xlabel('Prédit')
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{model_name}.png', dpi=150)
    plt.show()

    tn, fp, fn, tp = cm.ravel()
    print(f"\n{model_name} :")
    print(f"  TP : {tp}  |  TN : {tn}")
    print(f"  FP : {fp}  ← Fausse alarme")
    print(f"  FN : {fn}  ← Attaque manquée ⚠️")

plot_confusion_matrix(y_test, y_pred_cnn,  'CNN')
plot_confusion_matrix(y_test, y_pred_lstm, 'LSTM')

In [ ]:
print("=" * 55)
print("📊 CNN — Classification Report")
print("=" * 55)
print(classification_report(y_test, y_pred_cnn,
      target_names=['Normal', 'Attack']))

print("=" * 55)
print("📊 LSTM — Classification Report")
print("=" * 55)
print(classification_report(y_test, y_pred_lstm,
      target_names=['Normal', 'Attack']))

In [ ]:
def get_metrics(y_true, y_pred, model_name):
    return {
        'Modèle':    model_name,
        'Accuracy':  round(accuracy_score(y_true, y_pred)  * 100, 2),
        'Precision': round(precision_score(y_true, y_pred) * 100, 2),
        'Recall':    round(recall_score(y_true, y_pred)    * 100, 2),
        'F1-Score':  round(f1_score(y_true, y_pred)        * 100, 2),
    }

cnn_metrics  = get_metrics(y_test, y_pred_cnn,  'CNN')
lstm_metrics = get_metrics(y_test, y_pred_lstm, 'LSTM')

comparison_df = pd.DataFrame([cnn_metrics, lstm_metrics]).set_index('Modèle')

print("=" * 50)
print("📊 Tableau Comparatif CNN vs LSTM (%)")
print("=" * 50)
print(comparison_df.to_string())

comparison_df.T.plot(kind='bar', figsize=(10, 6),
                     color=['#3498db','#e74c3c'], edgecolor='black')
plt.title('Comparaison CNN vs LSTM', fontsize=13)
plt.ylabel('Score (%)')
plt.xticks(rotation=0)
plt.ylim(0, 115)
plt.legend(title='Modèle')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('comparison_cnn_lstm.png', dpi=150)
plt.show()

In [ ]:
cnn_model.save('cnn_model.h5')
lstm_model.save('lstm_model.h5')
np.save('y_pred_cnn.npy',  y_pred_cnn)
np.save('y_pred_lstm.npy', y_pred_lstm)

print("✅ Modèles sauvegardés !")
print("   → cnn_model.h5")
print("   → lstm_model.h5")
print("\n✅ Task 2 complète ! Prêt pour Task 3 🚀")

In [ ]:
!pip install openai -q
print("✅ OK !")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
warnings.filterwarnings('ignore')
from collections import Counter
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import load_model

print("✅ Imports OK !")

In [ ]:
lstm_model   = load_model('lstm_model.h5')
X_test       = np.load('X_test_final.npy')
y_test       = np.load('y_test_final.npy')
y_test_cat   = np.load('y_test_cat.npy')
feature_cols = np.load('feature_cols.npy', allow_pickle=True).tolist()
le           = joblib.load('label_encoder.pkl')

X_test_3d = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print("✅ Chargé !")
print(f"   Features : {len(feature_cols)}")
print(f"   Classes  : {le.classes_}")
print(f"   X_test   : {X_test.shape}")

In [ ]:
# Charger LSTM
lstm_model = load_model('lstm_model.h5')

# Charger données
X_test       = np.load('X_test_final.npy')
y_test       = np.load('y_test_final.npy')
y_test_cat   = np.load('y_test_cat.npy')
feature_cols = np.load('feature_cols.npy', allow_pickle=True).tolist()
le           = joblib.load('label_encoder.pkl')

X_test_3d = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print("✅ Modèle et données chargés !")
print(f"   Features : {len(feature_cols)}")
print(f"   Classes  : {le.classes_}")
print(f"   X_test   : {X_test.shape}")

In [ ]:
# Wrapper Keras → Sklearn
class LSTMWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, model):
        self.model = model
    def fit(self, X, y):
        return self
    def predict(self, X):
        X_3d = X.reshape(X.shape[0], X.shape[1], 1)
        return (self.model.predict(X_3d, verbose=0) > 0.5).astype(int).flatten()
    def score(self, X, y):
        return accuracy_score(y, self.predict(X))

# Sous-ensemble pour la vitesse
sample_size = 2000
idx      = np.random.choice(len(X_test), sample_size, replace=False)
X_sample = X_test[idx]
y_sample = y_test[idx]

wrapper = LSTMWrapper(lstm_model)

print("⏳ Calcul permutation importance...")
perm_result = permutation_importance(
    wrapper, X_sample, y_sample,
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

# Top 5 features
importances   = perm_result.importances_mean
top5_idx      = np.argsort(importances)[::-1][:5]
top5_features = [feature_cols[i] for i in top5_idx]
top5_scores   = [importances[i]  for i in top5_idx]

print("\n=== TOP 5 FEATURES GLOBALES ===")
for i, (feat, score) in enumerate(zip(top5_features, top5_scores)):
    print(f"  {i+1}. {feat:40s} : {score:.4f}")

# Visualisation
plt.figure(figsize=(10, 5))
plt.barh(top5_features[::-1], top5_scores[::-1],
         color=['#9b59b6','#2ecc71','#3498db','#f39c12','#e74c3c'])
plt.title('Top 5 Features — Permutation Importance (LSTM)', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

In [ ]:
# Prédictions
y_pred_prob = lstm_model.predict(X_test_3d, verbose=0).flatten()
y_pred      = (y_pred_prob > 0.5).astype(int)

# Index des classes
dos_idx   = le.transform(['DoS'])[0]
probe_idx = le.transform(['Probe'])[0]

# Cas correctement détectés
dos_correct   = np.where((y_test_cat == dos_idx)   & (y_pred == 1))[0]
probe_correct = np.where((y_test_cat == probe_idx) & (y_pred == 1))[0]

dos_case   = dos_correct[0]
probe_case = probe_correct[0]

print(f"✅ Cas DoS   : index {dos_case}")
print(f"✅ Cas Probe : index {probe_case}")

# Extraire valeurs top 5 features
def get_case_features(idx):
    case_data = {}
    for feat, score in zip(top5_features, top5_scores):
        feat_idx = feature_cols.index(feat)
        case_data[feat] = {
            'value':      round(float(X_test[idx][feat_idx]), 4),
            'importance': round(float(score), 4)
        }
    return case_data

dos_features   = get_case_features(dos_case)
probe_features = get_case_features(probe_case)

print("\n=== CAS DoS — Valeurs Top 5 Features ===")
for feat, info in dos_features.items():
    print(f"  {feat:40s} : {info['value']}")

print("\n=== CAS Probe — Valeurs Top 5 Features ===")
for feat, info in probe_features.items():
    print(f"  {feat:40s} : {info['value']}")

In [ ]:
# System Prompt SOC Analyst
SYSTEM_PROMPT = """You are a Senior SOC (Security Operations Center) Analyst
with 10+ years of experience in network intrusion detection and cybersecurity.
You analyze network traffic data and LSTM model outputs to generate professional
security alert reports.

For each alert, you must provide:
1. TECHNICAL EXPLANATION: Why these feature values triggered the alert
2. SEVERITY RATING: Low / Medium / High / Critical
3. MITIGATION STEPS: At least 3 immediate actions for the network administrator

Be precise, professional, and actionable in your analysis."""

print("✅ System Prompt défini !")
print(SYSTEM_PROMPT)

In [ ]:
def generate_alert_report(attack_type, features_dict, confidence):
    """
    Génère un rapport d'alerte SOC simulé basé sur les features LSTM
    """
    features_str = "\n".join([
        f"  - {feat}: {info['value']} (importance: {info['importance']})"
        for feat, info in features_dict.items()
    ])

    # Déterminer severity
    if confidence > 0.95:
        severity = "CRITICAL"
        color    = "🔴"
    elif confidence > 0.85:
        severity = "HIGH"
        color    = "🟠"
    elif confidence > 0.70:
        severity = "MEDIUM"
        color    = "🟡"
    else:
        severity = "LOW"
        color    = "🟢"

    # Templates par type d'attaque
    reports = {
        'DoS': {
            'technical': f"""The LSTM model detected anomalous network behavior
consistent with a Denial-of-Service (DoS) attack pattern.
Key indicators include elevated connection counts and error rates
suggesting flooding behavior aimed at exhausting server resources.

Feature Analysis:
{features_str}

The combination of high src_bytes, elevated serror_rate, and
abnormal count values indicates a deliberate attempt to overwhelm
the target system with malicious traffic.""",

            'mitigation': [
                "1. 🛡️  Immediately activate rate-limiting rules on the affected network segment",
                "2. 🚫  Block the source IP range at the perimeter firewall",
                "3. 📊  Enable enhanced traffic logging and packet capture for forensic analysis",
                "4. 🔔  Notify upstream ISP to implement null-routing if attack persists",
                "5. 🔄  Activate DDoS mitigation service (Cloudflare/Arbor) if available"
            ]
        },
        'Probe': {
            'technical': f"""The LSTM model identified network scanning and
reconnaissance activity consistent with a Probe attack.
The attacker appears to be systematically mapping the network
topology and discovering open services/vulnerabilities.

Feature Analysis:
{features_str}

The pattern of low-duration connections across multiple services
combined with elevated dst_host_count suggests automated port
scanning tools (e.g., Nmap) are being used to enumerate targets.""",

            'mitigation': [
                "1. 🔍  Immediately isolate and analyze the scanning source IP",
                "2. 🚧  Deploy honeypot systems to track and analyze attacker behavior",
                "3. 🔒  Review and tighten firewall rules — close unnecessary open ports",
                "4. 📝  Document all scanned IP ranges for threat intelligence reporting",
                "5. ⚠️  Escalate to Tier-2 SOC team — probe often precedes a larger attack"
            ]
        }
    }

    report = reports.get(attack_type, reports['DoS'])

    print(f"""
╔══════════════════════════════════════════════════════════════╗
║           🚨 SOC ALERT REPORT — {attack_type:10s}                  ║
╚══════════════════════════════════════════════════════════════╝

{color} SEVERITY    : {severity}
🎯 ATTACK TYPE : {attack_type}
📊 CONFIDENCE  : {confidence*100:.2f}%
⏰ TIMESTAMP   : {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📋 TECHNICAL EXPLANATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{report['technical']}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🛠️  MITIGATION STEPS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")
    for step in report['mitigation']:
        print(f"  {step}")

    print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
⚡ STATUS : ALERT GENERATED — AWAITING ANALYST REVIEW
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")
    return severity

# Générer rapport DoS
dos_confidence   = float(y_pred_prob[dos_case])
dos_severity     = generate_alert_report('DoS', dos_features, dos_confidence)

In [ ]:
# Générer rapport Probe
probe_confidence = float(y_pred_prob[probe_case])
probe_severity   = generate_alert_report('Probe', probe_features, probe_confidence)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# DoS features
dos_vals   = [info['value'] for info in dos_features.values()]
dos_feats  = [f[:20] for f in dos_features.keys()]
axes[0].barh(dos_feats, dos_vals, color='#e74c3c', edgecolor='black')
axes[0].set_title(f'CAS DoS — Valeurs Features\nConfidence: {dos_confidence*100:.1f}%',
                  fontsize=11)
axes[0].set_xlabel('Valeur (après scaling)')
axes[0].grid(True, alpha=0.3, axis='x')

# Probe features
probe_vals  = [info['value'] for info in probe_features.values()]
probe_feats = [f[:20] for f in probe_features.keys()]
axes[1].barh(probe_feats, probe_vals, color='#3498db', edgecolor='black')
axes[1].set_title(f'CAS Probe — Valeurs Features\nConfidence: {probe_confidence*100:.1f}%',
                  fontsize=11)
axes[1].set_xlabel('Valeur (après scaling)')
axes[1].grid(True, alpha=0.3, axis='x')

plt.suptitle('Interprétation Locale — DoS vs Probe', fontsize=13)
plt.tight_layout()
plt.savefig('local_interpretation.png', dpi=150)
plt.show()

In [ ]:
discussion = """
╔══════════════════════════════════════════════════════════════╗
║        💡 HUMAN-IN-THE-LOOP — DISCUSSION                    ║
╚══════════════════════════════════════════════════════════════╝

1. POURQUOI C'EST IMPORTANT ?
   → Le LSTM détecte → Le LLM explique → L'analyste décide
   → Réduit la fatigue d'alerte (alert fatigue) dans le SOC
   → Transforme un label binaire en intelligence actionnable

2. AVANTAGES DE L'APPROCHE XAI + LLM :
   ✅ Transparence    : On sait POURQUOI le modèle a alerté
   ✅ Rapidité        : Rapport généré en millisecondes
   ✅ Standardisation : Même format pour toutes les alertes
   ✅ Formation       : Les juniors apprennent des explications

3. LIMITES ET RISQUES :
   ⚠️  Hallucination LLM  : Le LLM peut générer des explications fausses
   ⚠️  Faux positifs      : LSTM imparfait → LLM amplifie l'erreur
   ⚠️  Sur-automatisation : Risque de faire confiance aveuglément

4. CONCLUSION :
   → L'humain reste indispensable pour valider les alertes critiques
   → Le système LSTM + LLM est un assistant, pas un remplaçant
"""
print(discussion)

In [ ]:
results = {
    'top5_features': dict(zip(top5_features, [float(s) for s in top5_scores])),
    'dos_case': {
        'index':      int(dos_case),
        'confidence': float(dos_confidence),
        'severity':   dos_severity,
        'features':   {k: v['value'] for k, v in dos_features.items()}
    },
    'probe_case': {
        'index':      int(probe_case),
        'confidence': float(probe_confidence),
        'severity':   probe_severity,
        'features':   {k: v['value'] for k, v in probe_features.items()}
    }
}

import json
with open('task3_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("✅ Résultats Task 3 sauvegardés !")
print(json.dumps(results, indent=2))
print("\n✅ Task 3 complète ! Prêt pour Task 4 🚀")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
import json
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime
from tensorflow.keras.models import load_model

print("✅ Imports OK !")

In [ ]:
lstm_model   = load_model('lstm_model.h5')
X_test       = np.load('X_test_final.npy')
y_test       = np.load('y_test_final.npy')
y_test_cat   = np.load('y_test_cat.npy')
feature_cols = np.load('feature_cols.npy', allow_pickle=True).tolist()
le           = joblib.load('label_encoder.pkl')

with open('task3_results.json', 'r') as f:
    task3_results = json.load(f)

X_test_3d = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print("✅ Tout chargé !")
print(f"   Top 5 features : {list(task3_results['top5_features'].keys())}")

In [ ]:
lstm_model   = load_model('lstm_model.h5')
X_test       = np.load('X_test_final.npy')
y_test       = np.load('y_test_final.npy')
y_test_cat   = np.load('y_test_cat.npy')
feature_cols = np.load('feature_cols.npy', allow_pickle=True).tolist()
le           = joblib.load('label_encoder.pkl')

with open('task3_results.json', 'r') as f:
    task3_results = json.load(f)

X_test_3d = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print("✅ Tout chargé !")
print(f"   Top 5 features : {list(task3_results['top5_features'].keys())}")

In [ ]:
# ── Matrice de décision ────────────────────────────────────────
DECISION_MATRIX = {
    'LOW': {
        'confidence_range': (0.50, 0.70),
        'action':           'ALERT_ONLY',
        'description':      'Log and continue monitoring',
        'color':            '#2ecc71',
        'emoji':            '🟢'
    },
    'MEDIUM': {
        'confidence_range': (0.70, 0.85),
        'action':           'FLAG_FOR_REVIEW',
        'description':      'Flag for human review + increase logging',
        'color':            '#f39c12',
        'emoji':            '🟡'
    },
    'HIGH': {
        'confidence_range': (0.85, 0.95),
        'action':           'BLOCK_IP',
        'description':      'Initiate automated blocking',
        'color':            '#e67e22',
        'emoji':            '🟠'
    },
    'CRITICAL': {
        'confidence_range': (0.95, 1.00),
        'action':           'ISOLATE_HOST',
        'description':      'Isolate host + emergency response',
        'color':            '#e74c3c',
        'emoji':            '🔴'
    }
}

# ── Infrastructure critique protégée ──────────────────────────
PROTECTED_IPS = [
    '192.168.1.1',    # Core Router
    '192.168.1.10',   # Database Server
    '192.168.1.20',   # Management Server
    '10.0.0.1',       # Firewall
    '10.0.0.254'      # Domain Controller
]

print("✅ Decision Matrix définie !")
for severity, config in DECISION_MATRIX.items():
    print(f"  {config['emoji']} {severity:10s} → {config['action']:20s} | {config['description']}")

In [ ]:
def safety_filter(destination_ip, action):
    """
    Vérifie si la destination est une infrastructure critique.
    Empêche toute action automatique sur les serveurs critiques.
    """
    if destination_ip in PROTECTED_IPS:
        return {
            'blocked':   True,
            'reason':    f"⚠️ SAFETY FILTER ACTIVATED — {destination_ip} is critical infrastructure",
            'action':    'HUMAN_REVIEW_REQUIRED',
            'original':  action
        }
    return {
        'blocked': False,
        'reason':  'IP not in protected range',
        'action':  action
    }

# Test du Safety Filter
print("=== TEST SAFETY FILTER ===")
test1 = safety_filter('192.168.1.10', 'BLOCK_IP')
print(f"IP Critique  : {test1}")

test2 = safety_filter('172.16.0.55', 'BLOCK_IP')
print(f"IP Normale   : {test2}")

In [ ]:
class AutonomousNIDSAgent:
    """
    Agent autonome NIDS — Pipeline LSTM → LLM → Action
    """
    def __init__(self, lstm_model, decision_matrix, protected_ips):
        self.lstm_model      = lstm_model
        self.decision_matrix = decision_matrix
        self.protected_ips   = protected_ips
        self.action_log      = []

    def determine_severity(self, confidence):
        """Détermine la sévérité basée sur le score de confiance"""
        for severity, config in self.decision_matrix.items():
            low, high = config['confidence_range']
            if low <= confidence < high:
                return severity
        return 'CRITICAL' if confidence >= 0.95 else 'LOW'

    def determine_attack_type(self, y_cat):
        """Détermine le type d'attaque depuis le label"""
        return le.classes_[y_cat]

    def safety_check(self, destination_ip, action):
        """Vérifie si l'action est sûre"""
        return safety_filter(destination_ip, action)

    def generate_llm_reasoning(self, attack_type, severity, confidence, features):
        """Simule le raisonnement LLM"""
        reasonings = {
            'DoS': f"High-confidence DoS pattern detected (conf={confidence:.2f}). "
                   f"Elevated connection rates and error ratios indicate flooding behavior. "
                   f"Automated response justified at {severity} severity.",

            'Probe': f"Network reconnaissance activity confirmed (conf={confidence:.2f}). "
                     f"Systematic port scanning detected across multiple hosts. "
                     f"Early-stage attack — blocking recommended to prevent escalation.",

            'R2L': f"Remote-to-Local intrusion attempt detected (conf={confidence:.2f}). "
                   f"Unauthorized access attempt from external source. "
                   f"Immediate isolation required to prevent data exfiltration.",

            'U2R': f"CRITICAL: User-to-Root privilege escalation detected (conf={confidence:.2f}). "
                   f"Attacker attempting to gain root access. "
                   f"Full host isolation mandatory — potential system compromise.",

            'Normal': f"Traffic classified as normal (conf={confidence:.2f}). "
                      f"No action required."
        }
        return reasonings.get(attack_type, f"Unknown attack type detected (conf={confidence:.2f}).")

    def execute(self, sample_idx, destination_ip='172.16.0.55'):
        """
        Pipeline complet : LSTM → Severity → Safety → Action
        """
        # ── Étape 1 : LSTM Detection ──────────────────────────
        x        = X_test[sample_idx].reshape(1, -1, 1)
        conf     = float(self.lstm_model.predict(x, verbose=0)[0][0])
        is_attack = conf > 0.5
        y_cat    = y_test_cat[sample_idx]
        attack_type = self.determine_attack_type(y_cat)

        # ── Étape 2 : Severity Assessment ─────────────────────
        if not is_attack:
            severity = 'LOW'
            action   = 'ALERT_ONLY'
        else:
            severity = self.determine_severity(conf)
            action   = self.decision_matrix[severity]['action']

        # ── Étape 3 : Safety Filter ───────────────────────────
        safety = self.safety_check(destination_ip, action)
        if safety['blocked']:
            action = safety['action']

        # ── Étape 4 : LLM Reasoning ───────────────────────────
        reasoning = self.generate_llm_reasoning(
            attack_type, severity, conf,
            task3_results.get(f'{attack_type.lower()}_case', {})
        )

        # ── Étape 5 : JSON Response ───────────────────────────
        response = {
            'timestamp':        datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'sample_index':     int(sample_idx),
            'destination_ip':   destination_ip,
            'lstm_confidence':  round(conf, 4),
            'is_attack':        bool(is_attack),
            'attack_type':      attack_type,
            'severity':         severity,
            'action_taken':     action,
            'safety_triggered': safety['blocked'],
            'reasoning':        reasoning
        }

        self.action_log.append(response)
        return response

    def print_response(self, response):
        """Affiche le rapport de l'agent"""
        config = self.decision_matrix.get(response['severity'],
                 self.decision_matrix['LOW'])
        emoji  = config.get('emoji', '⚪')

        print(f"""
┌─────────────────────────────────────────────────────────────┐
│  🤖 AUTONOMOUS AGENT RESPONSE                               │
├─────────────────────────────────────────────────────────────┤
│  ⏰ Timestamp    : {response['timestamp']}          │
│  🎯 Attack Type  : {response['attack_type']:10s}                        │
│  {emoji} Severity     : {response['severity']:10s}                        │
│  📊 Confidence   : {response['lstm_confidence']*100:.2f}%                          │
│  ⚡ Action Taken : {response['action_taken']:20s}              │
│  🛡️  Safety Filter: {'ACTIVATED ⚠️' if response['safety_triggered'] else 'PASSED ✅':20s}              │
├─────────────────────────────────────────────────────────────┤
│  💬 REASONING:                                              │
│  {response['reasoning'][:60]:60s}│
│  {response['reasoning'][60:120]:60s}│
└─────────────────────────────────────────────────────────────┘
""")

print("✅ Agent NIDS défini !")

In [ ]:
agent = AutonomousNIDSAgent(
    lstm_model      = lstm_model,
    decision_matrix = DECISION_MATRIX,
    protected_ips   = PROTECTED_IPS
)
print("✅ Agent initialisé et prêt !")

In [ ]:
print("=" * 65)
print("🧪 TEST CASE 1 — LOW Severity (Normal Traffic)")
print("=" * 65)

# Chercher un cas Normal
normal_idx = np.where(y_test == 0)[0][0]

response_low = agent.execute(
    sample_idx     = normal_idx,
    destination_ip = '172.16.0.100'
)
agent.print_response(response_low)
print("📄 JSON Response:")
print(json.dumps(response_low, indent=2))

In [ ]:
print("=" * 65)
print("🧪 TEST CASE 2 — HIGH Severity (DoS Attack)")
print("=" * 65)

# Chercher un cas DoS avec haute confiance
dos_idx_arr = np.where(y_test_cat == le.transform(['DoS'])[0])[0]
X_dos       = X_test[dos_idx_arr].reshape(len(dos_idx_arr), -1, 1)
dos_probs   = lstm_model.predict(X_dos, verbose=0).flatten()

# Trouver index avec confiance entre 0.85 et 0.95 (HIGH)
high_mask = (dos_probs >= 0.85) & (dos_probs < 0.95)
if high_mask.sum() > 0:
    high_sample = dos_idx_arr[np.where(high_mask)[0][0]]
else:
    high_sample = dos_idx_arr[np.argmax(dos_probs)]

response_high = agent.execute(
    sample_idx     = high_sample,
    destination_ip = '172.16.0.55'
)
agent.print_response(response_high)
print("📄 JSON Response:")
print(json.dumps(response_high, indent=2))

In [ ]:
print("=" * 65)
print("🧪 TEST CASE 2 — HIGH Severity (DoS Attack)")
print("=" * 65)

# Chercher un cas DoS avec haute confiance
dos_idx_arr = np.where(y_test_cat == le.transform(['DoS'])[0])[0]
X_dos       = X_test[dos_idx_arr].reshape(len(dos_idx_arr), -1, 1)
dos_probs   = lstm_model.predict(X_dos, verbose=0).flatten()

# Trouver index avec confiance entre 0.85 et 0.95 (HIGH)
high_mask = (dos_probs >= 0.85) & (dos_probs < 0.95)
if high_mask.sum() > 0:
    high_sample = dos_idx_arr[np.where(high_mask)[0][0]]
else:
    high_sample = dos_idx_arr[np.argmax(dos_probs)]

response_high = agent.execute(
    sample_idx     = high_sample,
    destination_ip = '172.16.0.55'
)
agent.print_response(response_high)
print("📄 JSON Response:")
print(json.dumps(response_high, indent=2))

In [ ]:
print("=" * 65)
print("🧪 TEST CASE 3 — CRITICAL Severity + Safety Filter")
print("=" * 65)

# Chercher cas avec très haute confiance
attack_idx  = np.where(y_test == 1)[0]
X_att       = X_test[attack_idx].reshape(len(attack_idx), -1, 1)
att_probs   = lstm_model.predict(X_att, verbose=0).flatten()
crit_sample = attack_idx[np.argmax(att_probs)]

# Destination = serveur critique → Safety Filter activé
response_critical = agent.execute(
    sample_idx     = crit_sample,
    destination_ip = '192.168.1.10'  # Database Server protégé
)
agent.print_response(response_critical)
print("📄 JSON Response:")
print(json.dumps(response_critical, indent=2))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 16))
ax.set_xlim(0, 10)
ax.set_ylim(0, 20)
ax.axis('off')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')

def draw_box(ax, x, y, w, h, text, color, fontsize=10):
    box = mpatches.FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle="round,pad=0.1",
        facecolor=color, edgecolor='#2c3e50',
        linewidth=2, zorder=3
    )
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold',
            color='white', zorder=4, wrap=True)

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#2c3e50',
                                lw=2), zorder=2)

# Boxes
draw_box(ax, 5, 19,   6, 1.0, '🌐 Network Traffic Input',    '#2c3e50', 11)
draw_box(ax, 5, 16.5, 6, 1.2, '🧠 LSTM Detection Engine\n(Binary Classification)', '#8e44ad', 10)
draw_box(ax, 5, 14,   6, 1.2, '📊 Confidence Score\n& Attack Type Assessment',     '#2980b9', 10)
draw_box(ax, 5, 11.5, 6, 1.2, '⚖️ Decision Matrix\n(LOW / MEDIUM / HIGH / CRITICAL)', '#16a085', 10)
draw_box(ax, 5, 9,    6, 1.2, '🛡️ Safety Filter\n(Critical Infrastructure Check)',  '#d35400', 10)
draw_box(ax, 5, 6.5,  6, 1.2, '💬 LLM Reasoning Engine\n(SOC Analyst Simulation)',  '#c0392b', 10)

# Action boxes
draw_box(ax, 1.5, 3.5, 2.2, 1.0, '🟢 ALERT\nONLY',       '#27ae60', 9)
draw_box(ax, 4,   3.5, 2.2, 1.0, '🟡 FLAG FOR\nREVIEW',  '#f39c12', 9)
draw_box(ax, 6.5, 3.5, 2.2, 1.0, '🟠 BLOCK\nIP',         '#e67e22', 9)
draw_box(ax, 9,   3.5, 2.2, 1.0, '🔴 ISOLATE\nHOST',     '#e74c3c', 9)

draw_box(ax, 5, 1.5, 6, 1.0, '📋 JSON Response + SOC Report', '#2c3e50', 11)

# Arrows
draw_arrow(ax, 5, 18.5, 5, 17.1)
draw_arrow(ax, 5, 15.9, 5, 14.6)
draw_arrow(ax, 5, 13.4, 5, 12.1)
draw_arrow(ax, 5, 10.9, 5, 9.6)
draw_arrow(ax, 5, 8.4,  5, 7.1)
draw_arrow(ax, 5, 5.9,  1.5, 4.0)
draw_arrow(ax, 5, 5.9,  4,   4.0)
draw_arrow(ax, 5, 5.9,  6.5, 4.0)
draw_arrow(ax, 5, 5.9,  9,   4.0)
draw_arrow(ax, 1.5, 3.0, 5, 2.0)
draw_arrow(ax, 4,   3.0, 5, 2.0)
draw_arrow(ax, 6.5, 3.0, 5, 2.0)
draw_arrow(ax, 9,   3.0, 5, 2.0)

# Labels
ax.text(5, 20, 'LSTM → LLM → AGENT Pipeline', ha='center',
        fontsize=14, fontweight='bold', color='#2c3e50')
ax.text(7, 5.2, 'Severity\nLevel', ha='center', fontsize=8, color='#7f8c8d')

plt.tight_layout()
plt.savefig('agent_pipeline_flowchart.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Flowchart sauvegardé !")

In [ ]:
all_responses = [response_low, response_high, response_critical]

print("=" * 70)
print("📊 RÉSUMÉ — 3 TEST CASES AGENT AUTONOME")
print("=" * 70)

summary_df = pd.DataFrame([{
    'Test Case':      f"Case {i+1}",
    'Attack Type':    r['attack_type'],
    'Confidence':     f"{r['lstm_confidence']*100:.1f}%",
    'Severity':       r['severity'],
    'Action':         r['action_taken'],
    'Safety Filter':  '⚠️ YES' if r['safety_triggered'] else '✅ NO'
} for i, r in enumerate(all_responses)])

print(summary_df.to_string(index=False))

# Visualisation
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')
table = ax.table(
    cellText    = summary_df.values,
    colLabels   = summary_df.columns,
    cellLoc     = 'center',
    loc         = 'center'
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 2)

colors_map = {'LOW':'#d5f5e3','MEDIUM':'#fef9e7','HIGH':'#fdebd0','CRITICAL':'#fadbd8'}
for i, r in enumerate(all_responses):
    color = colors_map.get(r['severity'], '#ffffff')
    for j in range(len(summary_df.columns)):
        table[(i+1, j)].set_facecolor(color)

plt.title('Agent Autonome — Résumé des Actions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('agent_summary.png', dpi=150)
plt.show()

In [ ]:
conclusion = """
╔══════════════════════════════════════════════════════════════╗
║     🤖 CONCLUSION — AUTONOMOUS RESPONSE CHALLENGES          ║
╚══════════════════════════════════════════════════════════════╝

1. ✅ AVANTAGES DE L'AGENT AUTONOME :
   → Temps de réponse : millisecondes vs minutes (humain)
   → Disponibilité    : 24/7 sans fatigue
   → Cohérence        : Même règle appliquée pour chaque alerte
   → Scalabilité      : Gère des milliers d'alertes simultanément

2. ⚠️  RISQUES ET DÉFIS :

   a) FAUX POSITIFS :
      → Un BLOCK_IP automatique peut couper un service légitime
      → Ex: Un scan interne de sécurité classifié comme Probe
      → Solution: Whitelist + période de grâce avant blocage

   b) FAUX NÉGATIFS :
      → Une attaque sophistiquée non détectée par le LSTM
      → L'agent ne réagit pas → breach silencieuse
      → Solution: Modèles ensembles + règles heuristiques backup

   c) ADVERSARIAL ATTACKS :
      → Un attaquant peut crafted des paquets pour tromper le LSTM
      → L'agent prend une mauvaise décision avec haute confiance
      → Solution: Adversarial training + détection d'anomalies

   d) INFRASTRUCTURE CRITIQUE :
      → Notre Safety Filter protège les IPs critiques
      → Mais un attaquant peut usurper une IP protégée
      → Solution: Multi-factor verification avant toute action

3. 🎯 RECOMMANDATIONS FINALES :
   → Garder TOUJOURS un Human-in-the-Loop pour CRITICAL
   → Auditer régulièrement les décisions de l'agent
   → Mettre à jour le modèle LSTM mensuellement
   → Tester le système avec Red Team exercises

4. 📊 PIPELINE FINAL :
   LSTM (Detection) → LLM (Reasoning) → Agent (Action)
   ↑_________________Feedback Loop________________________↑
"""
print(conclusion)

In [ ]:
task4_results = {
    'agent_config': {
        'decision_matrix': {k: v['action'] for k, v in DECISION_MATRIX.items()},
        'protected_ips':   PROTECTED_IPS
    },
    'test_cases': all_responses
}

with open('task4_results.json', 'w') as f:
    json.dump(task4_results, f, indent=2)

print("✅ Task 4 complète !")
print("   → agent_pipeline_flowchart.png")
print("   → agent_summary.png")
print("   → task4_results.json")
print("\n🎉 TOUTES LES TÂCHES COMPLÈTES !")
print("   Task 1 ✅ EDA + Preprocessing")
print("   Task 2 ✅ CNN + LSTM")
print("   Task 3 ✅ XAI + LLM")
print("   Task 4 ✅ Autonomous Agent")